# BM25 Search Baseline

This notebook establishes a lexical BM25 baseline for the product search ranking system.

BM25 retrieves products based on lexical overlap between the user's search query
and product text. This provides a strong traditional information-retrieval baseline
before moving to dense retrieval, hybrid search, and learning-to-rank.

The product catalog contains 124,428 products. Model development and evaluation
use query-level train/validation/test splits to prevent query leakage.

## Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from rank_bm25 import BM25Okapi

In [2]:
PROJECT_ROOT = Path.cwd().parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PRODUCT_CATALOG_PATH = PROCESSED_DIR / "product_catalog.parquet"
TRAIN_PATH = PROCESSED_DIR / "train.parquet"
VALIDATION_PATH = PROCESSED_DIR / "validation.parquet"
TEST_PATH = PROCESSED_DIR / "test.parquet"

In [3]:
product_catalog = pd.read_parquet(PRODUCT_CATALOG_PATH)
validation = pd.read_parquet(VALIDATION_PATH)

In [4]:
validation.shape

(7395, 5)

## Tokenizer
We will use characters n-gram to better capture context of query with typos and joint-words (such as deck over and deckover)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from rank_bm25 import BM25Okapi

# 1. Setup Mock DataFrame (Simulating your product catalog)
data = {
    'product_uid':,
    'product_title': ["Heavy Duty Deckover Trailer", "Flatbed Utility Trailer", "Standard Skateboard Deck"],
    'product_description': ["This structural flatbed deckover unit is built for hauling.", "Great for lawnmowers and small vehicles.", "Premium 8-inch maple wood skate deck."],
    'attribute_text': ["Type: Deckover Axles: 2", "Type: Flatbed Axles: 1", "Type: Deck Size: 8in"]
}
product_catalog = pd.DataFrame(data)

# Fill any NaN values with empty strings to avoid errors
for col in ['product_title', 'product_description', 'attribute_text']:
    product_catalog[col] = product_catalog[col].fillna("").astype(str)

# =====================================================================
# 2. Build the Character N-Gram Tokenizer Pipeline
# =====================================================================

# We combine all text columns to teach the vectorizer the global vocabulary.
# 'char_wb' creates n-grams *only* within word boundaries (crucial for words like "deckover")
combined_text = (
    product_catalog['product_title'] + " " + 
    product_catalog['product_description'] + " " + 
    product_catalog['attribute_text']
)

vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5))
vectorizer.fit(combined_text)

# This is our custom n-gram tokenizer function
ngram_tokenizer = vectorizer.build_analyzer()

# Test the tokenizer on a joined word example
# Output will look like: [' dec', 'dec', 'eck', 'cko', 'kov', 'ove', 'ver', 'er ']
# print(ngram_tokenizer("deckover")) 

# =====================================================================
# 3. Tokenize Fields and Initialize Separate BM25 Indexes
# =====================================================================

# Tokenize each field independently for every row
tokenized_titles = [ngram_tokenizer(text) for text in product_catalog['product_title']]
tokenized_descs = [ngram_tokenizer(text) for text in product_catalog['product_description']]
tokenized_attrs = [ngram_tokenizer(text) for text in product_catalog['attribute_text']]

# Fit 3 distinct BM25 models (one per field)
bm25_title_model = BM25Okapi(tokenized_titles)
bm25_desc_model = BM25Okapi(tokenized_descs)
bm25_attr_model = BM25Okapi(tokenized_attrs)

# =====================================================================
# 4. Feature Extraction Function for a Query
# =====================================================================

def get_bm25_field_scores(query: str):
    """
    Takes a raw string query, converts it to character n-grams,
    and returns separate BM25 scores for every product in the catalog.
    """
    # 1. Transform user query into the exact same character n-gram format
    tokenized_query = ngram_tokenizer(query)
    
    # 2. Generate raw BM25 scores for all documents in the catalog
    title_scores = bm25_title_model.get_scores(tokenized_query)
    desc_scores = bm25_desc_model.get_scores(tokenized_query)
    attr_scores = bm25_attr_model.get_scores(tokenized_query)
    
    # 3. Build a clean DataFrame mapped back to product_uid
    features_df = pd.DataFrame({
        'product_uid': product_catalog['product_uid'],
        'bm25_title': title_scores,
        'bm25_description': desc_scores,
        'bm25_attribute': attr_scores
    })
    
    return features_df

# =====================================================================
# 5. Test It with a Typo/Joint Word Query
# =====================================================================
user_query = "deck over" # Notice the space, whereas catalog has "deckover"
features_output = get_bm25_field_scores(user_query)

print(features_output)


In [5]:
def tokenize(text):
    return text.lower().split()

In [6]:
product_tokens = product_catalog["product_text"].map(tokenize)

In [7]:
bm25 = BM25Okapi(product_tokens)

In [9]:
product_catalog

,product_uid,product_title,product_description,attribute_text,product_text
0,100001,Simpson Strong-Tie 12-Gauge Angle,"Not only do angles make joints stronger, they ...",Bullet01: Versatile connector for various 90Â°...,Title: Simpson Strong-Tie 12-Gauge Angle. Desc...
1,100002,BEHR Premium Textured DeckOver 1-gal. #SC-141 ...,BEHR Premium Textured DECKOVER is an innovativ...,"Application Method: Brush,Roller,Spray. Assemb...",Title: BEHR Premium Textured DeckOver 1-gal. #...
2,100003,STERLING Ensemble 33-1/4 in. x 60 in. x 75-1/4...,Classic architecture meets contemporary design...,Built-in flange: Yes. Bullet01: Slightly narro...,Title: STERLING Ensemble 33-1/4 in. x 60 in. x...
3,100004,Grape Solar 265-Watt Polycrystalline Solar Pan...,The Grape Solar 265-Watt Polycrystalline PV So...,Amperage (amps): 8.56. Bullet01: Positive powe...,Title: Grape Solar 265-Watt Polycrystalline So...
4,100005,Delta Vero 1-Handle Shower Only Faucet Trim Ki...,Update your bathroom with the Delta Vero Singl...,Bath Faucet Type: Combo Tub and Shower. Built-...,Title: Delta Vero 1-Handle Shower Only Faucet ...
...,...,...,...,...,...
124423,224424,stufurhome Norma 24 in. W x 16 in. D x 34 in. ...,Create a neat yet stylish storage space for or...,Assembly Required: No. Bullet01: Solid wood co...,Title: stufurhome Norma 24 in. W x 16 in. D x ...
124424,224425,Home Decorators Collection 49 in. D Alessandro...,Our Bullnose Adirondack Chair Cushions fit Adi...,Assembled Depth (in.): 49 in. Assembled Height...,Title: Home Decorators Collection 49 in. D Ale...
124425,224426,Simpson Strong-Tie HB 3-1/2 x 14 in. Top Flang...,Joist hangers are designed to provide support ...,,Title: Simpson Strong-Tie HB 3-1/2 x 14 in. To...
124426,224427,1/4 in. -20 tpi x 1-1/2 in. Stainless Steel Bu...,These socket cap screws are ideal for applicat...,,Title: 1/4 in. -20 tpi x 1-1/2 in. Stainless S...


In [10]:
product_index = pd.Series(
    np.arange(len(product_catalog)),
    index=product_catalog["product_uid"]
)

In [12]:
def get_bm25_scores(query, product_uids):
    query_tokens = tokenize(query)

    indices = product_index.loc[product_uids].to_numpy()

    scores = bm25.get_scores(query_tokens)

    return scores[indices]

Below is an example of a computing the BM25 score for a single query

In [17]:
validation

,id,product_uid,product_title,search_term,relevance
0,9,100002,BEHR Premium Textured DeckOver 1-gal. #SC-141 ...,deck over,3.00
1,34,100010,Valley View Industries Metal Stakes (4-Pack),steele stake,2.67
2,69,100017,MD Building Products 36 in. x 36 in. Cloverlea...,door guards,1.00
3,117,100022,Samsung 4.2 cu. ft. Front Load Washer with Ste...,PLATFORM FOR WASHERS,2.67
4,120,100022,Samsung 4.2 cu. ft. Front Load Washer with Ste...,samsung front load washer 3.7,2.00
...,...,...,...,...,...
7390,221248,206464,American Standard Exposed Yoke Adjustable Roug...,rough in wall mount,2.67
7391,221256,206470,Coast HP21R Focusing Rechargeable 1326 Lumen L...,rechargeable flashlight with 100 lumen,2.67
7392,221261,206474,Feiss August Moon 1-Light Corinthian Bronze Ou...,decorative living room wall lighting,2.33
7393,221285,206494,American Pro Decor 5 in. x 5 in. x 6 in. Long ...,wood feature beams,2.00


In [18]:
query = validation["search_term"].iloc[0]

query_data = validation[
    validation["search_term"] == query
].copy()

scores = get_bm25_scores(
    query,
    query_data["product_uid"].tolist()
)

query_data["bm25_score"] = scores

query_data.sort_values(
    "bm25_score",
    ascending=False
)

,id,product_uid,product_title,search_term,relevance,bm25_score
544,17058,103015,Rust-Oleum Restore 4 gal. 10X Advanced Timberl...,deck over,1.67,6.924154
609,18899,103384,Rust-Oleum Restore 4 gal. 10X Advanced Cape Co...,deck over,2.33,6.914115
18,571,100092,Rust-Oleum Restore 1-gal. 4X Deck Coat,deck over,1.67,6.677194
550,17297,103057,Rust-Oleum Restore 5-gal. 4X Deck Coat,deck over,2.33,6.677194
0,9,100002,BEHR Premium Textured DeckOver 1-gal. #SC-141 ...,deck over,3.00,2.790315
301,9568,101629,BEHR Premium DeckOver 5-gal. #PFC-68 Silver Gr...,deck over,3.00,0.000000
330,10428,101795,BEHR Premium DeckOver 5-gal. #SC-145 Desert Sa...,deck over,2.67,0.000000
604,18775,103356,BEHR Premium DeckOver 5-gal. #SC-330 Redwood W...,deck over,2.67,0.000000


Note that BM25 is using `product_text` to compute the score, which contains the title, description as well as attributes of the product.

In [ ]:
from sklearn.metrics import ndcg_score

In [51]:
def calculate_ndcg(query_data, k):
    y_true = query_data["relevance"].to_numpy()
    y_score = query_data["bm25_score"].to_numpy()

    return ndcg_score(
        [y_true],
        [y_score],
        k=k
    )

In [52]:
print("NDCG@10:", calculate_ndcg(query_data, 10))
print("NDCG@20:", calculate_ndcg(query_data, 20))

NDCG@10: 0.8786732048269063
NDCG@20: 0.8786732048269063


In [17]:
results = []

for query, query_data in validation.groupby("search_term"):
    query_data = query_data.copy()

    query_data["bm25_score"] = get_bm25_scores(
        query,
        query_data["product_uid"].tolist()
    )

    results.append({
        "query": query,
        "num_products": len(query_data),
        "ndcg@10": calculate_ndcg(query_data, 10),
        "ndcg@20": calculate_ndcg(query_data, 20),
    })

bm25_results = pd.DataFrame(results)

In [33]:
bm25_results.head()

,query,num_products,ndcg@10,ndcg@20
0,.75 in pocket hole screws,5,0.930503,0.930503
1,"0.5mm bit, counter sink ground",4,0.892776,0.892776
2,1 1/4 pvcsch 80,9,0.936231,0.936231
3,1 in to 3/4 inch pvc elbow,4,0.978443,0.978443
4,1' pvc pipe,8,0.939936,0.939936


In [34]:
print(
    "Mean NDCG@10:",
    bm25_results["ndcg@10"].mean()
)

print(
    "Mean NDCG@20:",
    bm25_results["ndcg@20"].mean()
)

Mean NDCG@10: 0.9532332010921856
Mean NDCG@20: 0.9567022882702825


In [35]:
bm25_results[
    ["ndcg@10", "ndcg@20"]
].describe()

,ndcg@10,ndcg@20
count,1040.000000,1040.000000
mean,0.953233,0.956702
std,0.041255,0.037971
min,0.716711,0.798947
25%,0.933375,0.937771
50%,0.961386,0.964441
75%,0.985588,0.986895
max,1.000000,1.000000


In [21]:
def recall_at_k(retrieved_uids, relevant_uids, k):
    retrieved = set(retrieved_uids[:k])
    relevant = set(relevant_uids)

    if not relevant:
        return np.nan

    return len(retrieved & relevant) / len(relevant)

In [ ]:
relevance_counts = (
    validation
    .assign(is_relevant=validation["relevance"] >= 2.0)
    .groupby("search_term")["is_relevant"]
    .sum()
)

print(relevance_counts.describe())

print(
    "Queries with at least 1 relevant product:",
    (relevance_counts > 0).sum()
)

print(
    "Queries with 2+ relevant products:",
    (relevance_counts >= 2).sum()
)

print(
    "Queries with 3+ relevant products:",
    (relevance_counts >= 3).sum()
)

count    1040.000000
mean        6.001923
std         2.965311
min         0.000000
25%         4.000000
50%         6.000000
75%         8.000000
max        15.000000
Name: is_relevant, dtype: float64
Queries with at least 1 relevant product: 1031
Queries with 2+ relevant products: 995
Queries with 3+ relevant products: 891


In [23]:
def retrieve_top_k(query, k=100):
    query_tokens = tokenize(query)

    scores = bm25.get_scores(query_tokens)

    top_indices = np.argsort(scores)[::-1][:k]

    return product_catalog.iloc[top_indices]["product_uid"].to_numpy()

In [ ]:
from tqdm.auto import tqdm
import pickle


recall_results = []
retrieved_dict = {}

for query, query_data in tqdm(
    validation.groupby("search_term"),
    total=validation["search_term"].nunique()
):

    relevant_uids = query_data.loc[
        query_data["relevance"] >= 2.0,
        "product_uid"
    ].to_numpy()

    retrieved_uids = retrieve_top_k(query, k=100)
    retrieved_dict[query] = retrieved_uids.tolist()

    recall_results.append({
        "query": query,
        "num_relevant": len(relevant_uids),
        "recall@10": recall_at_k(
            retrieved_uids, relevant_uids, 10
        ),
        "recall@20": recall_at_k(
            retrieved_uids, relevant_uids, 20
        ),
        "recall@50": recall_at_k(
            retrieved_uids, relevant_uids, 50
        ),
        "recall@100": recall_at_k(
            retrieved_uids, relevant_uids, 100
        ),
    })

with open(PROCESSED_DIR / "bm25_retrieved_dict.pkl", "wb") as f:
    pickle.dump(retrieved_dict, f)

  0%|          | 0/1040 [00:00<?, ?it/s]